# Store Sales Time Series Forecasting
**CatBoost-based sales forecasting using historical sales, promotion, calendar, and oil-price features.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
oil_csv = pd.read_csv(
    "/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv",
    usecols=["date", "dcoilwtico"]
)

oil_prices = oil_csv[["date", "dcoilwtico"]].copy()

oil_prices["date"] = pd.to_datetime(oil_prices["date"])

oil_prices = oil_prices.rename(
    columns={"dcoilwtico": "oil_price"}
)

oil_prices = (
    oil_prices
    .sort_values("date")
    .set_index("date")
)

oil_prices["oil_price"] = (
    oil_prices["oil_price"]
    .interpolate(method="time")
    .bfill()
    .ffill()
)

oil_prices = oil_prices.reset_index()

oil_prices.head()

In [ ]:
train_csv = pd.read_csv(
    "/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv",
    usecols=[
        "id",
        "date",
        "store_nbr",
        "family",
        "onpromotion",
        "sales"
    ],
    parse_dates=["date"]
)

test_csv = pd.read_csv(
    "/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv",
    usecols=[
        "id",
        "date",
        "store_nbr",
        "family",
        "onpromotion"
    ],
    parse_dates=["date"]
)

print("Train shape:", train_csv.shape)
print("Test shape:", test_csv.shape)

In [ ]:
oil_for_join = (
    oil_prices[["date", "oil_price"]]
    .sort_values("date")
)


def add_previous_oil_price(data):
    data = data.copy()

    data["_row_order"] = np.arange(len(data))
    data["_original_index"] = data.index

    data = data.sort_values("date")

    data = pd.merge_asof(
        data,
        oil_for_join,
        on="date",
        direction="backward"
    )

    data = data.sort_values("_row_order")

    data = data.set_index("_original_index")
    data.index.name = None

    return data.drop(columns="_row_order")


train_df = add_previous_oil_price(train_csv)
test_df = add_previous_oil_price(test_csv)


train_df = train_df[
    [
        "id",
        "date",
        "store_nbr",
        "family",
        "onpromotion",
        "oil_price",
        "sales"
    ]
]

test_df = test_df[
    [
        "id",
        "date",
        "store_nbr",
        "family",
        "onpromotion",
        "oil_price"
    ]
]


print(
    f"Missing oil prices in train: "
    f"{train_df['oil_price'].isna().sum()}"
)

print(
    f"Missing oil prices in test: "
    f"{test_df['oil_price'].isna().sum()}"
)

In [ ]:
# Keep the final 15 days of the labeled data
# as a chronological validation set.

test_mask = train_df["date"].between(
    "2017-08-01",
    "2017-08-15"
)

train_data = train_df.loc[~test_mask].copy()
test_data = train_df.loc[test_mask].copy()


def make_features(data):

    features = data[
        [
            "date",
            "store_nbr",
            "family",
            "onpromotion",
            "oil_price"
        ]
    ].copy()

    features["store_nbr"] = features["store_nbr"].astype(str)

    features["year"] = features["date"].dt.year
    features["month"] = features["date"].dt.month
    features["day"] = features["date"].dt.day
    features["day_of_week"] = features["date"].dt.dayofweek

    features["days_since_start"] = (
        features["date"] - train_df["date"].min()
    ).dt.days

    return features.drop(columns="date")


# Training data
X_train = make_features(train_data)
y_train = train_data["sales"]

# Validation data
X_test = make_features(test_data)
y_test = test_data["sales"]

# Kaggle competition test data
X_forecast = make_features(test_df)

cat_features = [
    "store_nbr",
    "family"
]


print(
    f"Train dates: "
    f"{train_data['date'].min().date()} "
    f"to {train_data['date'].max().date()}"
)

print(
    f"Test dates: "
    f"{test_data['date'].min().date()} "
    f"to {test_data['date'].max().date()}"
)

print(
    f"Forecast dates: "
    f"{test_df['date'].min().date()} "
    f"to {test_df['date'].max().date()}"
)

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")
print(f"X_forecast: {X_forecast.shape}")

In [ ]:
X_train.info()

In [ ]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=200,
    learning_rate=0.05,
    depth=10,
    loss_function="RMSE",
    eval_metric="RMSE",
    verbose=50
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)

In [ ]:
y_pred = np.maximum(
    model.predict(X_test),
    0
)

y_forecast = np.maximum(
    model.predict(X_forecast),
    0
)


rmsle = np.sqrt(
    np.mean(
        (
            np.log1p(y_pred)
            - np.log1p(
                np.maximum(
                    y_test.to_numpy(),
                    0
                )
            )
        ) ** 2
    )
)

print(f"RMSLE: {rmsle:.6f}")

In [ ]:
def export_submission(
    y_forecast,
    file_name="/kaggle/working/submission.csv"
):

    submission = test_df[["id"]].copy()

    submission["sales"] = np.asarray(y_forecast)

    submission = (
        submission
        .sort_values("id")
        .reset_index(drop=True)
    )

    submission.to_csv(
        file_name,
        index=False
    )

    return submission


y_forecast = model.predict(X_forecast)

submission = export_submission(y_forecast)

submission.head()